# SignSense AI — MLP Training (Kaggle)

**Model:** MLP landmark classifier — ASL A–Z + space/del/nothing (29 classes)  
**Target accuracy:** > 95% | **Runtime:** ~15 min on Kaggle T4 GPU

### Before you start
1. Settings → Accelerator → **GPU T4 x2**
2. Add dataset: **+ Add Data** → search `grassknoted/asl-alphabet` → Add
3. Run all cells top to bottom

### Dataset path on Kaggle
The ASL dataset will be at: `/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train/`

In [1]:
# ── Cell 1: Setup paths ───────────────────────────────────────────────────────
import os

WORKING_DIR   = '/kaggle/working'
MODELS_DIR    = f'{WORKING_DIR}/models'
BACKEND_PATH  = f'{WORKING_DIR}/backend'
LOGS_DIR      = f'{WORKING_DIR}/logs/mlp'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

# Kaggle input dataset path (after adding grassknoted/asl-alphabet)
KAGGLE_INPUT  = '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train'

print(f'Models will be saved to: {MODELS_DIR}')
print(f'Dataset path: {KAGGLE_INPUT}')
print(f'Dataset exists: {os.path.exists(KAGGLE_INPUT)}')

Models will be saved to: /kaggle/working/models
Dataset path: /kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train
Dataset exists: False


In [3]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# NOTE: Kaggle requires internet to be enabled in Settings (right sidebar)
# Settings → Internet → ON
import subprocess, sys

# Check if mediapipe is already installed (Kaggle pre-installs some packages)
try:
    import mediapipe as mp
    print(f'✅ mediapipe already installed: {mp.__version__}')
except ImportError:
    print('Installing mediapipe (requires internet enabled in Settings)...')
    !pip install -q 'protobuf>=5.28.0' 'mediapipe>=0.10.18'
    import mediapipe as mp
    print(f'✅ mediapipe installed: {mp.__version__}')

!pip install -q scikit-learn tqdm albumentations
print('✅ All dependencies ready.')

✅ Dependencies installed.


ERROR: Invalid requirement: "'protobuf": Expected package name at the start of dependency specifier
    'protobuf
    ^


In [ ]:
# ── Cell 3: Clone project repo from GitHub ────────────────────────────────────
import os, sys

REPO_URL = 'https://github.com/prateek1756/sign-language-detection.git'

if not os.path.exists(f'{WORKING_DIR}/sign-language-detection'):
    !git clone {REPO_URL} {WORKING_DIR}/sign-language-detection
else:
    !git -C {WORKING_DIR}/sign-language-detection pull

BACKEND_PATH = f'{WORKING_DIR}/sign-language-detection/backend'
sys.path.insert(0, BACKEND_PATH)

from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'✅ Repo ready — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

In [ ]:
# ── Cell 4: Copy dataset + preprocess ────────────────────────────────────────
import os, shutil, numpy as np
from pathlib import Path

RAW_ASL_DIR   = f'{BACKEND_PATH}/data/raw/ASL'
PROCESSED_DIR = f'{BACKEND_PATH}/data/processed/ASL'
PROCESSED_NPY = f'{PROCESSED_DIR}/landmarks_all.npy'
LABELS_NPY    = f'{PROCESSED_DIR}/labels_all.npy'

if os.path.exists(PROCESSED_NPY):
    print('✅ Preprocessed data already exists — skipping.')
else:
    # Verify Kaggle dataset is mounted
    if not os.path.exists(KAGGLE_INPUT):
        raise FileNotFoundError(
            f'Dataset not found at {KAGGLE_INPUT}\n'
            'Add the dataset: + Add Data → search grassknoted/asl-alphabet → Add'
        )

    # Copy class directories to project raw data folder
    print(f'Copying dataset from {KAGGLE_INPUT}...')
    os.makedirs(RAW_ASL_DIR, exist_ok=True)
    class_dirs = [d for d in Path(KAGGLE_INPUT).iterdir() if d.is_dir()]
    print(f'Found {len(class_dirs)} class directories')

    for class_dir in class_dirs:
        dest = Path(RAW_ASL_DIR) / class_dir.name
        if not dest.exists():
            shutil.copytree(str(class_dir), str(dest))

    total = sum(len(list(d.glob('*.jpg'))) for d in Path(RAW_ASL_DIR).iterdir() if d.is_dir())
    print(f'✅ {total:,} images copied to {RAW_ASL_DIR}')

    print('\nRunning preprocessing pipeline (~10-20 min)...')
    !python {BACKEND_PATH}/src/preprocess.py --all --augment --aug_factor 3

    if not os.path.exists(PROCESSED_NPY):
        raise RuntimeError('Preprocessing failed. Check output above.')

X = np.load(PROCESSED_NPY)
y = np.load(LABELS_NPY)
print(f'\n✅ Data ready: X={X.shape}  y={y.shape}  classes={len(set(y.tolist()))}')

In [ ]:
# ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('✅ GPU memory growth enabled.')
else:
    print('WARNING: No GPU. Training will be slow.')

In [ ]:
# ── Cell 6: Train MLP ─────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'training_config', 'configs']):
        del sys.modules[mod]

from configs.training_config import MLPConfig
from src.train import train_mlp

cfg = MLPConfig()
cfg.save_dir = Path(MODELS_DIR)
cfg.log_dir  = Path(LOGS_DIR)
cfg.mixed_precision = True

print(f'  hidden_dims:    {cfg.hidden_dims}')
print(f'  epochs:         {cfg.epochs}')
print(f'  batch_size:     {cfg.batch_size}')
print(f'  learning_rate:  {cfg.learning_rate}')
print()

model = train_mlp(cfg)
print('\n✅ Training complete!')

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if 'evaluate' in mod:
        del sys.modules[mod]

import src.evaluate as ev
ev.MODELS_DIR = Path(MODELS_DIR)

results = ev.evaluate('asl_mlp', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: Verify saved model ────────────────────────────────────────────────
import os, numpy as np, tensorflow as tf

print('Files saved to working directory:')
for f in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

loaded = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'asl_mlp.keras'))
dummy  = np.zeros((1, 63), dtype=np.float32)
pred   = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f}')
print('\n✅ Model verified. Download asl_mlp.keras from the Output tab.')